# CarDD Baseline Model

**Pipeline position:** 03 — Baseline training and visual inspection.

**Dependency:** Run `02_cardd_annotation_conversion_clean.ipynb` first. This notebook consumes its YOLO **segmentation** output; the conversion is intentionally not repeated here.

## 1. Imports and Configuration

In [ ]:
from pathlib import Path
from collections import defaultdict
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from ultralytics import YOLO


In [ ]:
# This path matches PROCESSED_ROOT in Notebook 02.
PROCESSED_ROOT = Path("../data/cardd/processed").resolve()
DATA_YAML = PROCESSED_ROOT / "cardd.yaml"

print("Processed dataset:", PROCESSED_ROOT)
print("Dataset YAML:", DATA_YAML)
print("YAML exists:", DATA_YAML.exists())

if not DATA_YAML.exists():
    raise FileNotFoundError("Run Notebook 02 first so that cardd.yaml and the converted dataset are created.")


## 2. Load Dataset Configuration

In [ ]:
import yaml

with open(DATA_YAML, "r", encoding="utf-8") as f:
    dataset_config = yaml.safe_load(f)

class_names = {int(k): v for k, v in dataset_config["names"].items()}
print("Classes:")
for class_id, name in class_names.items():
    print(f"  {class_id}: {name}")


## 3. Validate YOLO Segmentation Labels

In [ ]:
def validate_segmentation_split(split):
    labels_dir = PROCESSED_ROOT / "labels" / split
    images_dir = PROCESSED_ROOT / "images" / split

    total_lines = 0
    invalid = []
    class_counts = {i: 0 for i in class_names}

    for label_file in labels_dir.glob("*.txt"):
        for line_no, line in enumerate(label_file.read_text(encoding="utf-8").splitlines(), 1):
            if not line.strip():
                continue
            parts = line.split()
            total_lines += 1

            # Segmentation label: class_id followed by x,y pairs.
            if len(parts) < 7 or (len(parts) - 1) % 2 != 0:
                invalid.append((label_file.name, line_no, "invalid polygon length"))
                continue

            try:
                values = list(map(float, parts))
                cls = int(values[0])
                coords = values[1:]
                if values[0] != cls or cls not in class_names:
                    invalid.append((label_file.name, line_no, "invalid class"))
                    continue
                if not all(0 <= x <= 1 for x in coords):
                    invalid.append((label_file.name, line_no, "coordinates outside [0,1]"))
                    continue
                class_counts[cls] += 1
            except ValueError:
                invalid.append((label_file.name, line_no, "non-numeric value"))

    print(f"\n{split.upper()}")
    print("Images:", len(list(images_dir.glob("*"))))
    print("Label files:", len(list(labels_dir.glob("*.txt"))))
    print("Polygon lines:", total_lines)
    print("Invalid lines:", len(invalid))
    for cls, count in class_counts.items():
        print(f"  {cls}: {class_names[cls]:15s} -> {count}")

    return invalid

validation_errors = {split: validate_segmentation_split(split) for split in ["train", "val", "test"]}


## 4. Visual Validation — One Sample per Class

In [ ]:
def images_by_class(split="train"):
    labels_dir = PROCESSED_ROOT / "labels" / split
    images_dir = PROCESSED_ROOT / "images" / split
    result = {cls: [] for cls in class_names}

    for label_file in labels_dir.glob("*.txt"):
        classes = set()
        for line in label_file.read_text(encoding="utf-8").splitlines():
            parts = line.split()
            if parts:
                try:
                    cls = int(float(parts[0]))
                    if cls in class_names:
                        classes.add(cls)
                except ValueError:
                    pass
        image_matches = list(images_dir.glob(label_file.stem + ".*"))
        if image_matches:
            for cls in classes:
                result[cls].append(image_matches[0])
    return result

def draw_segmentation(image_path, label_path, ax):
    image = np.array(Image.open(image_path).convert("RGB"))
    h, w = image.shape[:2]
    ax.imshow(image)

    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) < 7 or (len(parts) - 1) % 2 != 0:
            continue
        cls = int(float(parts[0]))
        coords = np.array(list(map(float, parts[1:])), dtype=float).reshape(-1, 2)
        pts = coords * np.array([w, h])
        polygon = np.vstack([pts, pts[0]])
        ax.plot(polygon[:, 0], polygon[:, 1], linewidth=2)
        ax.text(pts[0, 0], pts[0, 1], class_names[cls], fontsize=9)

    ax.axis("off")

random.seed(42)
by_class = images_by_class("train")

for cls, name in class_names.items():
    if not by_class[cls]:
        print(f"No samples found for {name}")
        continue
    image_path = random.choice(by_class[cls])
    label_path = PROCESSED_ROOT / "labels" / "train" / f"{image_path.stem}.txt"
    fig, ax = plt.subplots(figsize=(8, 6))
    draw_segmentation(image_path, label_path, ax)
    ax.set_title(f"{name} | {image_path.name}")
    plt.tight_layout()
    plt.show()


## 5. Bounding-Box Area Analysis from Segmentation Polygons

In [ ]:
# Approximate bounding-box area of each polygon in normalized coordinates.
areas = defaultdict(list)
labels_dir = PROCESSED_ROOT / "labels" / "train"

for label_file in labels_dir.glob("*.txt"):
    for line in label_file.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) < 7 or (len(parts) - 1) % 2 != 0:
            continue
        cls = int(float(parts[0]))
        coords = np.array(list(map(float, parts[1:])), dtype=float).reshape(-1, 2)
        x_min, y_min = coords.min(axis=0)
        x_max, y_max = coords.max(axis=0)
        areas[cls].append((x_max - x_min) * (y_max - y_min))

for cls, name in class_names.items():
    values = np.array(areas[cls])
    if len(values) == 0:
        continue
    print(f"{name:15} | n={len(values):4} | min={values.min():.6f} | median={np.median(values):.6f} | mean={values.mean():.6f} | 75%={np.percentile(values,75):.6f} | max={values.max():.6f}")


## 6. Baseline Training

In [ ]:
# GPU check
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Notebook 02 produces YOLO segmentation labels, so use the segmentation checkpoint.
model = YOLO("yolo11n-seg.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=10,
    imgsz=640,
    batch=4,
    device=0,
    workers=2,
    amp=False,
    project="runs/cardd",
    name="baseline_seg_10ep",
    exist_ok=True
)


## 7. Baseline Validation

In [ ]:
metrics = model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=640,
    batch=4,
    device=0
)
print(metrics)


## 8. Handoff / Next Step

In [ ]:
print("Baseline training and validation complete.")
print("Use the saved Ultralytics run under runs/cardd for training curves and evaluation results.")
